# Experiment 4: uncertainty-aware missing-feature MIA against DP targets

Question: when some candidate fields are hidden, does reconstruction-score
variability reveal membership beyond the mean score, and how does this change
across DP budgets? This combines Exp 4 accuracy-focused target training with
Exp 3's missing-feature protocol. It does not retrain targets on incomplete data.

**Data and targets.** ACS 2023 income prediction (nominal PINCP > 50,000),
48 inputs, 100,000 target training records and unchanged held-out partitions.
One fresh non-private control plus epsilon = 0.5, 1, 2, 5, 10, 25, 50;
constant delta = 1e-6. Architecture, regularization, Adam learning rate,
public-validation accuracy selection and the 30-epoch cap match Exp 4 accuracy.
Private training is clipped, noisy DP-Adam through Opacus, not ordinary Adam.

**Attacker.** A simulated black-box API returns both income-class probabilities
for completed native-field inputs. It permits repeated and blind-walk queries.
The attacker knows the income class for the loss feature, mask, feature schema
and public auxiliary data; it does not access target weights, gradients or
protected training records. Target membership labels are used only for evaluation;
known shadow memberships train and calibrate the auxiliary attack.
All target/shadow training uses complete schemas, with ordinary preprocessing
for naturally missing survey entries. Fields are hidden only at attack time.

**Frozen auxiliary pipeline.** Keep Exp 4's 16 non-private ACS 2014 shadows:
four RF, four gradient boosting, four XGBoost and four generic 128/64 MLPs.
Each fits preprocessing on its training split only. Shadows and the base RF
attack are trained once. Reserved 2014 donors fit seven conditional reconstruction
chains. Disjoint meta/calibration/test record groups fit and assess the aggregators.
Nothing is refitted or recalibrated using a DP target or its membership outcomes.

**Controls per mask.** Single marginal PFV fill; conditional first draw;
average of K=6 scores; RF on the mean; matched RF on mean and standard deviation.
The four conditional methods share exactly the same draws. Complete-feature
MIA is measured on the same target IDs. The main uncertainty test compares
mean+std against mean-only, rather than attributing every gain over PFV to std.

Both shadow and target walks retain Exp 4's consistent numeric protocol,
including JWMNP and JWRIP. This differs from Exp 3's historical commuting-field
walk asymmetry. The shadow pool and target training also differ from Exp 3, so
its historical 62.625% result is not a matched control for this notebook.

Fit auxiliary components once, then train/query/release each target in turn.
Use a separate artifact root. No old metrics replace fresh measurements, and
neither source notebook nor its outputs is modified. No checkpoints, saved
models, raw completion scores, private prediction tables or logs are written.
The full run is expensive: seven masks times six completions for every target
and held-out shadow observation, with up to 41 record queries per completion.
Creating this notebook does not execute or authorize the experiment.

## Dependencies and fixed study configuration
Use the project virtual environment. If the dependency check below fails, install
`python -m pip install -r requirements.txt` in that environment before running.
Installation is separate from executing this notebook.

Private training uses Opacus per-example gradient clipping and Gaussian noise
with Adam (DP-Adam), Poisson sampling and an RDP accountant. Physical batches
are capped at 32 to limit per-sample-gradient memory; logical batches remain 256.
The existing four paper-inspired attack features, including blind walks, are
retained with consistent numeric perturbations for shadows and targets.

In [ ]:
from pathlib import Path
from importlib.util import find_spec
import csv
import gc
import importlib.metadata
import json
import math

missing_dependencies = [name for name in ['opacus', 'matplotlib', 'xgboost'] if find_spec(name) is None]
if missing_dependencies:
    raise ImportError('Missing ' + ', '.join(missing_dependencies)
        + '. Install in this notebook kernel: python -m pip install -r requirements.txt')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from src.data import split_acs_data, subset_target_training, TARGET
import src.expanded_data as expanded_data_module
importlib.reload(expanded_data_module)
from src.expanded_data import EXPANDED_FEATURES, load_expanded_year, reference_split_config
from src.dp_experiment import (
    prepare_public_preprocessing, train_dp_study_target, plot_budget_accuracy,
)
from src.training import evaluate_model
import src.shadow_models as shadow_models_module
importlib.reload(shadow_models_module)
from src.shadow_models import (train_shadow_model, summarize_shadow_models,
    PAPER_SHADOW_ALGORITHMS, MLP_SHADOW_CONFIG)
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import collect_shadow_paper_features, collect_target_paper_features
from src.notebook_output import CellProgress, quiet_output, suppress_sklearn_parallel_warning

suppress_sklearn_parallel_warning()
torch.set_num_threads(4)
SEED = 42
TARGET_YEAR = 2023
AUXILIARY_YEAR = 2014
TRAIN_SIZE = 100_000
PUBLIC_PREPROCESSING_SIZE = 10_000
EPSILON_BUDGETS = (0.5, 1, 2, 5, 10, 25, 50)
DELTA = 1e-6
assert 0 < DELTA < 1 / TRAIN_SIZE
assert len(EXPANDED_FEATURES) == 48
N_SHADOW_MODELS = 16
SHADOW_RECORDS_PER_CLASS = 2000
SHADOW_ALGORITHMS = PAPER_SHADOW_ALGORITHMS + ('mlp',)
SHADOW_MLP_CONFIG = dict(MLP_SHADOW_CONFIG)
assert N_SHADOW_MODELS == 4 * len(SHADOW_ALGORITHMS)
SPLIT_CONFIG = reference_split_config()
assert SPLIT_CONFIG['seed'] == SEED
TRAIN_CONFIG = dict(hidden_dims=(512, 256, 128), dropout=0.1,
    max_epochs=30, patience=5, min_delta=0.0, batch_size=256,
    max_physical_batch_size=32, learning_rate=3e-4, weight_decay=1e-4,
    max_grad_norm=1.0, seed=SEED, device='cpu',
    selection_metric='accuracy', strict_accounting=True)
WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
    batch_size=512, seed=SEED)
EXPANDED_NUMERIC_WALK = {'JWMNP': (2.0, 0, 200), 'JWRIP': (1.0, 1, 10)}
RUN_DIR = Path('artifacts/exp_4_2023_2014_dp_uncertainty_mf_mia_k6_s16_m7')

for year, filename in [(TARGET_YEAR, 'psam_p06.csv'), (AUXILIARY_YEAR, 'ss14pca.csv')]:
    raw_path = Path('data/raw/folktables') / str(year) / '1-Year' / filename
    with raw_path.open(newline='', encoding='utf-8-sig') as handle:
        columns = set(next(csv.reader(handle)))
    missing_columns = set(EXPANDED_FEATURES + ['PINCP', 'PWGTP']) - columns
    if missing_columns:
        raise ValueError(f'{raw_path}: missing columns {sorted(missing_columns)}')

versions = {name: importlib.metadata.version(name) for name in
    ['torch', 'opacus', 'numpy', 'pandas', 'scikit-learn', 'xgboost', 'matplotlib']}
print('Budgets:', EPSILON_BUDGETS, '| Constant delta:', DELTA)
print('Epoch cap:', TRAIN_CONFIG['max_epochs'], '| Patience:', TRAIN_CONFIG['patience'])
print('Complete-feature and missing-feature MIA; frozen auxiliary-only reconstruction and aggregation.')
display(pd.Series(versions, name='package_version'))

from src.paper_attack_features import make_probability_oracle
from src.dp_missing_feature import score_missing_feature_target, uncertainty_contrasts
from src.uncertainty_mf_mia import (
    ConditionalDonorChain, auxiliary_roles, completion_scores, summarize_scores,
    fit_aggregation, evaluate_aggregation,
)
import time

UNCERTAINTY_K = 6
RECONSTRUCTION_SEED = SEED + 1000
MASK_FIELDS = ['MAR', 'WKHP', 'COW', 'SEX']
MISSING_MASKS = {'MAR': ['MAR'], 'WKHP': ['WKHP'], 'MAR+COW': ['MAR', 'COW'], 'COW+SEX': ['COW', 'SEX'], 'MAR+WKHP+SEX': ['MAR', 'WKHP', 'SEX'], 'WKHP+COW+SEX': ['WKHP', 'COW', 'SEX'], 'MAR+WKHP+COW+SEX': ['MAR', 'WKHP', 'COW', 'SEX']}
assert UNCERTAINTY_K == 6 and len(MISSING_MASKS) == 7
UNCERTAINTY_DIR = RUN_DIR / 'uncertainty_mf_mia'

## Public preprocessing and fixed target partitions
Make the same stratified target splits and 100,000-record training subset as
Exp 3. For this research experiment, the fixed 2023 validation partition is
explicitly **public**, outside the protected target training set; it selects
epochs only. Test and MIA nonmember records never select models or thresholds.
The privacy statement is conditional on this fixed partition and concerns
record-level target-training membership, not person-level or survey-selection privacy.

Reserve 10,000 public 2014 rows for preprocessing, disjoint from shadow records.
Learn all imputation/scaling statistics, missing-indicator choices and category
vocabularies on those public rows only, then freeze them for all eight targets.
There is no target-fitted preprocessing. The auxiliary data and validation data
are assumed available independently of the protected training records. Native
category codes remain year-specific; public 2014 vocabulary may not cover all
2023 categories. This limitation applies equally to every target here.

In [ ]:
target_source = load_expanded_year(TARGET_YEAR, income_base_year=None)
reference_partitions = split_acs_data(target_source, **SPLIT_CONFIG)
target_partitions = subset_target_training(reference_partitions, train_size=TRAIN_SIZE, seed=SEED)
for name in ['validation', 'test', 'mia_members', 'mia_nonmembers']:
    np.testing.assert_array_equal(target_partitions[name].index, reference_partitions[name].index)
assert len(target_partitions['train']) == TRAIN_SIZE
assert len(target_partitions['mia_members']) == len(target_partitions['mia_nonmembers'])

auxiliary_source = load_expanded_year(AUXILIARY_YEAR, income_base_year=None)
shadow_auxiliary, public_preprocessing = train_test_split(auxiliary_source,
    test_size=PUBLIC_PREPROCESSING_SIZE, stratify=auxiliary_source[TARGET], random_state=SEED)
shadow_auxiliary = shadow_auxiliary.sort_index().copy()
public_preprocessing = public_preprocessing.sort_index().copy()
assert set(shadow_auxiliary.index).isdisjoint(public_preprocessing.index)
assert set(auxiliary_source.index).isdisjoint(target_source.index)
target_preprocessor, target_arrays = prepare_public_preprocessing(public_preprocessing, target_partitions)
partition_counts = {name: len(frame) for name, frame in target_partitions.items()}
display(pd.Series({**partition_counts, 'public_2014_preprocessing': len(public_preprocessing),
    'public_2014_shadow_pool': len(shadow_auxiliary)}, name='records'))
print('Encoded input dimensions shared by all targets:', target_arrays['train']['X'].shape[1])
del target_source, reference_partitions, auxiliary_source
gc.collect()

## Fit the complete-feature auxiliary attack once
Train 16 public 2014 shadows: four each of RF, gradient boosting, XGBoost
and generic MLP. Existing tree settings and 40/20/40 splits are retained.
MLP configuration is recorded in SHADOW_MLP_CONFIG; only its shadow
validation partition selects epochs, never nonmembers or target records. Each shadow fits its own
training-only preprocessor. Query 2,000 members and 2,000 nonmembers per shadow.
Keep repeated auxiliary IDs in one grouped attack train/validation/test role.
Train one RF membership classifier and freeze its auxiliary-selected thresholds
before evaluating any target. Keep only each fitted model/preprocessor until auxiliary completion scoring
finishes; release them before training DP targets.

This is a **frozen transfer-attack evaluation** across privacy budgets. The
shadows are not DP MLPs matched to each budget. A weak transferred attack alone
does not establish resistance to an attacker adapted to DP training.

In [ ]:
def fit_and_query_shadow(auxiliary, name, algorithm, seed):
    with quiet_output():
        result = train_shadow_model(auxiliary, algorithm, seed=seed, schema='expanded_acs',
            parameters=SHADOW_MLP_CONFIG if algorithm == 'mlp' else None)
        result['attack_data']['shadow_model'] = name
        features, audits = collect_shadow_paper_features({name: result}, auxiliary,
            per_class=SHADOW_RECORDS_PER_CLASS, config=WALK_CONFIG, seed=SEED,
            include_blind_walk=True, numeric_protocol=EXPANDED_NUMERIC_WALK)
    summary = summarize_shadow_models({name: result})
    return features, audits[name], summary, dict(model=result['model'], preprocessor=result['preprocessor'])

shadow_results = {}
shadow_progress = CellProgress()
shadow_feature_parts, shadow_summary_parts, shadow_query_audits, shadow_seeds = [], [], {}, {}
for index in range(N_SHADOW_MODELS):
    algorithm = SHADOW_ALGORITHMS[index % len(SHADOW_ALGORITHMS)]
    name = f'{algorithm}_{index + 1:02d}'
    shadow_seeds[name] = SEED + 10 * index
    shadow_progress.update(f'Shadow {index + 1}/{N_SHADOW_MODELS}: training/querying {name}')
    features, audit, summary, fitted = fit_and_query_shadow(shadow_auxiliary, name, algorithm, shadow_seeds[name])
    shadow_results[name] = fitted
    shadow_feature_parts.append(features)
    shadow_summary_parts.append(summary)
    shadow_query_audits[name] = audit
    del features, audit, summary, fitted
    gc.collect()

shadow_attack_data = pd.concat(shadow_feature_parts, ignore_index=True)
shadow_income_metrics = pd.concat(shadow_summary_parts)
assert len(shadow_attack_data) == N_SHADOW_MODELS * 2 * SHADOW_RECORDS_PER_CLASS
assert shadow_attack_data['shadow_model'].nunique() == N_SHADOW_MODELS
with quiet_output():
    attack_result = train_membership_classifier(shadow_attack_data, seed=SEED,
        algorithm='random_forest', feature_set='paper')
split_ids = {name: set(frame['record_id']) for name, frame in attack_result['splits'].items()}
assert split_ids['train'].isdisjoint(split_ids['validation'] | split_ids['test'])
assert split_ids['validation'].isdisjoint(split_ids['test'])
auxiliary_attack_metrics = pd.DataFrame(attack_result['metrics']).T
shadow_progress.update('All shadows scored; auxiliary attack and thresholds frozen')
display(auxiliary_attack_metrics[['accuracy', 'balanced_accuracy', 'roc_auc', 'tpr_at_1pct_fpr_roc']].round(4))
del shadow_feature_parts, shadow_summary_parts
gc.collect()

## Auxiliary-only reconstruction and uncertainty calibration
The 10,000 public preprocessing records remain outside the shadow and donor pool.
Use the remaining 2014 records only. Base-attack training IDs never enter the
conditional meta/calibration/test tables. Repeated IDs retain a single role
across shadows. Donors exclude all base-attack/meta/calibration/test IDs.
Half the base validation IDs fit aggregators; the other half calibrate them.
The base auxiliary test IDs remain held out until evaluation.

Seven predeclared masks cover two singles, two pairs, two triples and all four
of MAR, WKHP, COW, SEX. Related visible predictors remain available. Conditional
chains use the same forest-leaf donor procedure as Exp 3, with no income or
membership conditioning. The sampled distribution is approximate, not a calibrated
posterior. Income remains available separately for the base loss attack feature.

In [ ]:
# Set roles once, before any target model is trained or queried.
acs_auxiliary = shadow_auxiliary  # Same public 2014 pool; no second data load.
assert len(shadow_results) == N_SHADOW_MODELS
assert set(shadow_attack_data['shadow_model']) == set(shadow_results)
uncertainty_roles = auxiliary_roles(acs_auxiliary, attack_result, seed=SEED)
assert set().union(*uncertainty_roles.values()) <= set(shadow_auxiliary.index)
assert set().union(*uncertainty_roles.values()).isdisjoint(public_preprocessing.index)
recon_fit_ids, recon_check_ids = train_test_split(
    sorted(uncertainty_roles['reconstruction']), test_size=0.2, random_state=SEED)
recon_training = acs_auxiliary.loc[recon_fit_ids, EXPANDED_FEATURES]
recon_check = acs_auxiliary.loc[recon_check_ids, EXPANDED_FEATURES]
uncertainty_reconstructors = {}
display(pd.Series({name: len(ids) for name, ids in uncertainty_roles.items()}, name='unique_2014_records'))

### Fit and audit conditional reconstruction using 2014 only
Held-out reconstruction checks use only reserved 2014 records, not attack candidates.
Numeric coverage is descriptive empirical coverage of the sampled 90% interval.
Categorical sample-match fraction measures draw accuracy; it is not calibrated
probability accuracy. Report these checks even when attack performance is poor.
Related fields remain visible, so reconstruction may be easy through proxies.

In [ ]:
# 2. Fit one conditional reconstruction chain per mask; no target queries
from src.notebook_output import CellProgress, quiet_output
reconstruction_audit_rows = []
reconstruction_progress = CellProgress()
for mask_index, (mask_name, hidden) in enumerate(MISSING_MASKS.items(), start=1):
    mask_started = time.monotonic()
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: fitting reconstruction')
    with quiet_output():
        reconstructor = ConditionalDonorChain(hidden, seed=RECONSTRUCTION_SEED).fit(recon_training)
    uncertainty_reconstructors[mask_name] = reconstructor
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: auditing {UNCERTAINTY_K} completions')
    with quiet_output():
        samples = list(reconstructor.iter_samples(recon_check,
            seeds=[RECONSTRUCTION_SEED + draw for draw in range(UNCERTAINTY_K)]))
    for field in hidden:
        observed = recon_check[field].notna().to_numpy()
        if not observed.any():
            continue
        draws = np.column_stack([sample[field].to_numpy()[observed] for sample in samples])
        truth = recon_check[field].to_numpy()[observed]
        row = dict(mask=mask_name, field=field, records=int(observed.sum()))
        if field == 'WKHP':
            lower, upper = np.quantile(draws, [0.05, 0.95], axis=1)
            row.update(mae_of_sample_mean=float(np.abs(draws.mean(axis=1) - truth).mean()),
                sampled_90pct_coverage=float(((truth >= lower) & (truth <= upper)).mean()),
                mean_interval_width=float((upper - lower).mean()))
        else:
            row['sample_match_fraction'] = float((draws == truth[:, None]).mean())
        reconstruction_audit_rows.append(row)
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: complete ({time.monotonic() - mask_started:.1f}s)')
reconstruction_audit = pd.DataFrame(reconstruction_audit_rows)
display(reconstruction_audit.round(4))
# Export this final audit in the final-results cell.

### Generate held-out auxiliary completion scores
The original 2,000 member/nonmember samples per shadow are reused. Only meta-fit,
calibration and auxiliary-test records are queried here; base-attack training
records are excluded. Repeated IDs retain the same role across all shadows.
Completion never receives membership labels as a conditioning variable.

Use Exp 4's numeric commuting-field protocol for these shadow queries, just as
for complete-feature shadows and every later DP/non-private target query.
The auxiliary scoring work is performed once, not once per privacy budget.

In [ ]:
# 3. K scores per held-out shadow observation, separately for every mask
from src.notebook_output import CellProgress, quiet_output
uncertainty_auxiliary_tables = {}
uncertainty_query_audits = {}
auxiliary_progress = CellProgress()
heldout_ids = set().union(*(uncertainty_roles[role] for role in ['meta', 'calibration', 'test']))
for mask_name, reconstructor in uncertainty_reconstructors.items():
    mask_tables = []
    for shadow_name, shadow in shadow_results.items():
        selected = shadow_attack_data.loc[
            (shadow_attack_data['shadow_model'] == shadow_name)
            & shadow_attack_data['record_id'].isin(heldout_ids),
            ['record_id', 'membership']].sort_values('record_id').reset_index(drop=True)
        if selected.empty:
            continue
        assert selected['record_id'].is_unique
        candidates = acs_auxiliary.loc[selected['record_id']]
        oracle = make_probability_oracle(shadow['model'], shadow['preprocessor'], schema='acs')
        auxiliary_progress.update(f'Auxiliary scoring: {mask_name} / {shadow_name}, {len(selected):,} records, K={UNCERTAINTY_K}')
        with quiet_output():
            draw_scores, audit = completion_scores(oracle, candidates, reconstructor,
                attack_result['classifier'], k=UNCERTAINTY_K, seed=RECONSTRUCTION_SEED,
                walk_config=WALK_CONFIG, include_blind_walk=True, numeric_protocol=EXPANDED_NUMERIC_WALK)
        table = summarize_scores(draw_scores)
        table['record_id'] = selected['record_id'].to_numpy()
        table['membership'] = selected['membership'].to_numpy()
        table['shadow_model'] = shadow_name
        table['role'] = [next(role for role in ['meta', 'calibration', 'test']
            if record_id in uncertainty_roles[role]) for record_id in table['record_id']]
        mask_tables.append(table)
        uncertainty_query_audits[f'{mask_name}/{shadow_name}'] = audit
    uncertainty_auxiliary_tables[mask_name] = pd.concat(mask_tables, ignore_index=True)
auxiliary_progress.update(f'Auxiliary scoring complete: {len(uncertainty_auxiliary_tables)} masks')

In [ ]:
# 4. Fit aggregators, freeze thresholds on calibration, evaluate auxiliary test
from src.notebook_output import quiet_output
uncertainty_aggregators = {}
uncertainty_auxiliary_result_rows = []
for mask_name, table in uncertainty_auxiliary_tables.items():
    meta = table.loc[table['role'] == 'meta']
    calibration = table.loc[table['role'] == 'calibration']
    auxiliary_test = table.loc[table['role'] == 'test']
    with quiet_output():
        learned, thresholds = fit_aggregation(meta, calibration, seed=SEED)
    uncertainty_aggregators[mask_name] = (learned, thresholds)
    with quiet_output():
        metrics, predictions = evaluate_aggregation(auxiliary_test, learned, thresholds)
    uncertainty_auxiliary_result_rows.extend(metrics.assign(mask=mask_name,
        n_missing=len(MISSING_MASKS[mask_name]), dataset='2014_auxiliary_test').to_dict('records'))
print('Auxiliary test: macro averages across masks of the same size')
display(pd.DataFrame(uncertainty_auxiliary_result_rows).groupby(['n_missing', 'method'])[
    ['roc_auc', 'balanced_accuracy', 'tpr_at_1pct_fpr_roc']].mean().round(4))

### Freeze the attack and release shadow models
Single and average score cutoffs are selected on conditional calibration scores.
The two matched RFs (300 trees, depth 12, minimum leaf 20) are fitted on the
meta group, using mean alone or mean+std, then calibrated separately. PFV and
complete-feature MIA retain the base attack's original validation thresholds.
Each cutoff stays fixed across all target privacy budgets. Target membership
and test results never select methods, masks, thresholds, epochs, epsilon or K.

In [ ]:
uncertainty_auxiliary_results = pd.DataFrame(uncertainty_auxiliary_result_rows)
assert len(uncertainty_aggregators) == len(MISSING_MASKS)
assert set(uncertainty_aggregators) == set(uncertainty_reconstructors) == set(MISSING_MASKS)
frozen_thresholds = {mask: thresholds for mask, (_, thresholds) in uncertainty_aggregators.items()}
# Keep only public reconstruction/aggregation models and the base classifier.
# No DP target has been trained yet. No retraining of shadows will be needed.
del shadow_results, shadow, oracle, uncertainty_auxiliary_tables
del recon_training, recon_check, samples, draws, draw_scores, table, meta, calibration, auxiliary_test
gc.collect()
print('Auxiliary reconstruction, aggregators and thresholds frozen for every target condition.')

## Train and evaluate one target at a time
The helper below defines the work; the following run cell performs it. Each
model starts from scratch with the same architecture/initialization seed and
regularization. Opacus uses independent, nondeterministically seeded sampling
and noise generators; their seeds are not saved. Highest public-validation
accuracy selects the returned epoch; lower log loss breaks exact ties. The test set, private training metrics and MIA
labels do not influence training or model selection.

Each private run calibrates noise for the **fixed maximum 30-epoch schedule**.
The fixed-schedule epsilon bound covers the early-stopped, validation-selected
model. Epsilon at the actual stopping time is recorded separately as a diagnostic;
it is not advertised as a tighter adaptive-stopping guarantee. Restoring an
earlier epoch does not reset the privacy expenditure. Delta remains 1e-6.

After selection, measure train/test utility and the signed train-minus-test
accuracy gap. Query the same balanced target MIA records using the frozen
four-feature attack. Only one target model and its query table are held at a time.

Before the first update, recalibrate noise using the private loader's exact
logical step count and the same conservative rate used by the Opacus accountant.
Record both the actual sampler rate and the accountant/bound rate. Public
preprocessing, splits, architecture, regularization, auxiliary attack and blind
walks follow the original DP experiment. Learning rate, epoch selection,
accounting calibration, expanded budget grid and mixed shadow composition differ
in this follow-up. Target training settings are unaffected by adding MLP shadows.

Before releasing each model, query complete-feature MIA and the five missing-feature
methods. All methods use exactly the original balanced member/nonmember IDs.
The same sorted candidates, reconstruction seeds and K draws are used across
privacy budgets, reducing additional completion randomness. This does not remove
DP training randomness. Only final metrics and query-count audits leave the helper.

In [ ]:
target_progress = CellProgress()

def run_target_condition(epsilon):
    condition = 'non_private' if epsilon is None else f'epsilon_{epsilon:g}'
    target_progress.update(f'{condition}: training from scratch')
    with quiet_output():
        model, history, metadata = train_dp_study_target(
            target_arrays['train']['X'], target_arrays['train']['y'],
            target_arrays['validation']['X'], target_arrays['validation']['y'],
            epsilon=epsilon, delta=DELTA, progress=target_progress.update, **TRAIN_CONFIG)
    target_progress.update(f'{condition}: evaluating selected epoch {metadata["best_epoch"]}')
    income_metrics = {name: evaluate_model(model, target_arrays[name]['X'], target_arrays[name]['y'])
        for name in ['train', 'validation', 'test']}
    target_progress.update(f'{condition}: querying complete-feature membership attack')
    with quiet_output():
        attack_features, query_audit = collect_target_paper_features(model, target_preprocessor,
            target_partitions, config=WALK_CONFIG, seed=SEED,
            include_blind_walk=True, numeric_protocol=EXPANDED_NUMERIC_WALK)
        mia_metrics, _ = evaluate_attack(attack_result['classifier'], attack_features,
            threshold=attack_result['accuracy_threshold'],
            low_fpr_threshold=attack_result['low_fpr_threshold'])
    assert mia_metrics['records'] == 2 * mia_metrics['members']
    row = dict(condition=condition, private=epsilon is not None,
        epsilon_requested=epsilon, delta=DELTA if epsilon is not None else None,
        epsilon_fixed_schedule_bound=metadata['epsilon_fixed_schedule_bound'],
        epsilon_at_stop_diagnostic=metadata['epsilon_at_stop_diagnostic'],
        noise_multiplier=metadata['noise_multiplier'],
        sampler_sample_rate=metadata['sampler_sample_rate'],
        accountant_sample_rate=metadata['accountant_sample_rate'],
        bound_sample_rate=metadata['bound_sample_rate'],
        selection_metric=metadata['selection_metric'], epochs_run=metadata['epochs_run'],
        best_epoch=metadata['best_epoch'], training_seconds=metadata['training_seconds'],
        target_train_accuracy=income_metrics['train']['accuracy'],
        target_validation_accuracy=income_metrics['validation']['accuracy'],
        target_test_accuracy=income_metrics['test']['accuracy'],
        target_test_roc_auc=income_metrics['test']['roc_auc'],
        target_test_log_loss=income_metrics['test']['log_loss'],
        generalization_gap=income_metrics['train']['accuracy'] - income_metrics['test']['accuracy'],
        mia_accuracy=mia_metrics['accuracy'], mia_balanced_accuracy=mia_metrics['balanced_accuracy'],
        mia_roc_auc=mia_metrics['roc_auc'], mia_tpr_at_1pct_fpr_roc=mia_metrics['tpr_at_1pct_fpr_roc'],
        mia_tpr_at_shadow_threshold=mia_metrics['tpr_at_shadow_threshold'],
        mia_fpr_at_shadow_threshold=mia_metrics['fpr_at_shadow_threshold'])
    target_progress.update(f'{condition}: scoring seven missing-feature masks')
    query = make_probability_oracle(model, target_preprocessor, schema='acs')
    with quiet_output():
        missing_rows, missing_audits = score_missing_feature_target(
            query, {name: target_partitions[name] for name in ['mia_members', 'mia_nonmembers']},
            shadow_auxiliary, MISSING_MASKS, uncertainty_reconstructors, uncertainty_aggregators,
            attack_result, k=UNCERTAINTY_K, reconstruction_seed=RECONSTRUCTION_SEED,
            pfv_seed=SEED, walk_config=WALK_CONFIG, numeric_protocol=EXPANDED_NUMERIC_WALK,
            progress=lambda message: target_progress.update(f'{condition}: {message}'))
    complete_row = dict(mask='complete', n_missing=0, method='complete', **mia_metrics)
    attack_rows = [complete_row] + missing_rows
    for attack_row in attack_rows:
        attack_row.update(condition=condition, private=epsilon is not None,
            epsilon_requested=epsilon, delta=DELTA if epsilon is not None else None,
            epsilon_fixed_schedule_bound=metadata['epsilon_fixed_schedule_bound'])
    metadata.update(income_metrics=income_metrics, membership_metrics=mia_metrics,
        query_audit=query_audit, missing_feature_query_audits=missing_audits)
    # Return metrics only: no fitted target, private row scores or model files.
    return row, history, metadata, attack_rows

## Run the eight matched target conditions
This cell starts target training and repeated attacks; execute only with explicit
approval. All earlier shadow/reconstruction/scoring cells are also experiments.
Train and evaluate a fresh non-private target plus every declared private budget.
No historical scores, checkpoints or fitted models are loaded. Each model is
released after its complete-feature and missing-feature scores have been reduced
to final metrics. Do not rerun this cell to choose a favorable noise realization.

In [ ]:
# Reload presentation support only; keep existing data, shadows and attack.
import importlib
import src.notebook_output as notebook_output_module
importlib.reload(notebook_output_module)
from src.notebook_output import compact_dp_warnings

study_rows, training_histories, target_metadata, missing_feature_rows = [], {}, {}, []
with compact_dp_warnings():
    for epsilon in (None,) + EPSILON_BUDGETS:
        row, history, metadata, attack_rows = run_target_condition(epsilon)
        missing_feature_rows.extend(attack_rows)
        study_rows.append(row)
        training_histories[row['condition']] = history
        target_metadata[row['condition']] = metadata
        gc.collect()
study_results = pd.DataFrame(study_rows)
assert len(study_results) == 1 + len(EPSILON_BUDGETS)
assert study_results.loc[study_results['private'], 'delta'].eq(DELTA).all()
target_progress.update('All eight conditions completed; final metrics ready')

missing_feature_results = pd.DataFrame(missing_feature_rows)
assert len(missing_feature_results) == len(study_results) * (1 + 5 * len(MISSING_MASKS))
assert not missing_feature_results.duplicated(['condition', 'mask', 'method']).any()
assert missing_feature_results['records'].eq(2 * len(target_partitions['mia_members'])).all()
assert missing_feature_results.loc[missing_feature_results['private'], 'delta'].eq(DELTA).all()
uncertainty_changes = uncertainty_contrasts(missing_feature_results)

## Compare utility, generalization and membership leakage
Accuracies below are fractions; graphs show percentages. The generalization gap
is training accuracy minus test accuracy. A small gap with poor test accuracy can
indicate underfitting, so both are reported. Ordinary MIA accuracy uses one
auxiliary-validation-selected threshold, frozen across targets. ROC-AUC and
TPR@1%FPR supplement accuracy so threshold transfer is visible.

For missing features, inspect every mask and method in the detailed CSV.
Mean+std minus mean-only is the paired descriptive contrast for uncertainty.
Mean results over masks summarize the selected study; masks are not independent
replicates, and those means are not a statistical significance test.
TPR@1%FPR derived from each target ROC is a retrospective evaluation statistic.
Also report TPR and realized FPR at the auxiliary-selected cutoff, which can
differ substantially from 1% after distribution shift or DP training.

In [ ]:
display(study_results.set_index('condition')[[
    'epsilon_requested', 'epsilon_fixed_schedule_bound', 'epsilon_at_stop_diagnostic',
    'delta', 'epochs_run', 'best_epoch', 'target_train_accuracy', 'target_test_accuracy',
    'generalization_gap', 'mia_accuracy', 'mia_roc_auc', 'mia_tpr_at_1pct_fpr_roc'
]].round(6))

display(study_results.loc[study_results['private'], [
    'condition', 'sampler_sample_rate', 'accountant_sample_rate', 'bound_sample_rate',
    'noise_multiplier', 'epsilon_fixed_schedule_bound'
]].set_index('condition'))

print('Missing-feature target metrics: descriptive macro averages over the seven masks')
display(missing_feature_results.loc[missing_feature_results['method'] != 'complete'].groupby(
    ['condition', 'method'])[['accuracy', 'balanced_accuracy', 'roc_auc',
        'tpr_at_1pct_fpr_roc', 'tpr_at_shadow_threshold', 'fpr_at_shadow_threshold']].mean().round(4))
print('Per-mask uncertainty minus learned-mean contrasts (positive favors uncertainty):')
display(uncertainty_changes.set_index(['condition', 'mask'])[
    ['accuracy_change', 'roc_auc_change', 'tpr_at_1pct_fpr_roc_change']].round(4))

## Graph 1: target model accuracy versus privacy budget
Use held-out **income-prediction test accuracy**. Epsilon labels are categories
with equal spacing, not numeric coordinates. The dashed line is the matched
non-private control. Budget labels describe requested maximum-schedule budgets;
actual accountant values and stopping epochs are available in the result table.

In [ ]:
figure, axes = plot_budget_accuracy(study_results, EPSILON_BUDGETS,
    metric='target_test_accuracy', output_dir=RUN_DIR, delta=DELTA)
axes.set_ylim(0.70, 0.90)
for suffix in ('png', 'svg'):
    figure.savefig(RUN_DIR / f'target_accuracy_vs_epsilon.{suffix}', dpi=200)
display(figure)
plt.close(figure)

## Graph 2: membership inference accuracy versus privacy budget
Use complete-feature **target MIA accuracy**, with the same categorical epsilon
axis. The non-private control and 50% chance level are horizontal references.
Chance is 50% because members and nonmembers are equally represented. Neither
curve is forced to be monotonic. One run per condition does not support error
bars or a claim that a near-chance attack proves privacy.

In [ ]:
figure, axes = plot_budget_accuracy(study_results, EPSILON_BUDGETS,
    metric='mia_accuracy', output_dir=RUN_DIR, delta=DELTA)
axes.set_ylim(0.40, 0.60)
for suffix in ('png', 'svg'):
    figure.savefig(RUN_DIR / f'mia_accuracy_vs_epsilon.{suffix}', dpi=200)
display(figure)
plt.close(figure)

## Missing-feature curves for every mask
Draw accuracy, ROC-AUC and TPR@1%FPR in separate per-mask panels, with all
five missing-feature methods and the complete-feature reference. Epsilon is
categorical; the last category is the fresh non-private control. Lines connect
categories for readability, not as a fitted monotonic relationship. Axes expand
to include every measured point. The same record IDs are used for each curve.

In [ ]:
from matplotlib.ticker import PercentFormatter

def plot_missing_feature_curves(metric, title, filename):
    methods = ['complete', 'pfv', 'single', 'average', 'learned_mean', 'uncertainty']
    names = ['Complete features', 'Marginal PFV', 'Conditional single',
        'Conditional average', 'RF mean only', 'RF mean + std']
    colors = ['black', 'tab:gray', 'tab:orange', 'tab:green', 'tab:blue', 'tab:red']
    conditions = [f'epsilon_{value:g}' for value in EPSILON_BUDGETS] + ['non_private']
    labels = [f'epsilon {value:g}' for value in EPSILON_BUDGETS] + ['Non-private']
    fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True, sharey=True, layout='constrained')
    lower = float(missing_feature_results[metric].min())
    upper = float(missing_feature_results[metric].max())
    limits = (max(0, min(.40, lower - .02)), min(1, max(.60, upper + .02)))
    if metric == 'tpr_at_1pct_fpr_roc':
        limits = (0, min(1, max(.05, upper + .01)))
    for ax, mask in zip(axes.flat, MISSING_MASKS):
        for method, name, color in zip(methods, names, colors):
            selected = missing_feature_results.loc[
                (missing_feature_results['method'] == method) &
                (missing_feature_results['mask'] == ('complete' if method == 'complete' else mask))]
            values = selected.set_index('condition').reindex(conditions)[metric]
            assert values.notna().all(), f'Incomplete curve: {mask} / {method}'
            ax.plot(range(len(labels)), values.to_numpy(), marker='o', markersize=3,
                linewidth=1.5, color=color, label=name)
        ax.set(title=mask, ylim=limits)
        ax.set_xticks(range(len(labels)), labels, rotation=50, ha='right', fontsize=8)
        ax.tick_params(axis='x', labelbottom=True)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
        ax.grid(axis='y', alpha=.2)
        if metric in ['accuracy', 'roc_auc']:
            ax.axhline(.5, color='.5', linewidth=1, linestyle=':')
    for ax in list(axes.flat)[len(MISSING_MASKS):]:
        ax.set_visible(False)
    handles, legend_labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc='outside lower center', ncol=6, fontsize=9)
    fig.suptitle(f'{title}; constant delta = {DELTA:g}; K = {UNCERTAINTY_K}')
    for suffix in ['png', 'svg']:
        fig.savefig(RUN_DIR / f'{filename}.{suffix}', dpi=200)
    display(fig)
    plt.close(fig)

for metric, title, filename in [
    ('accuracy', 'Missing-feature membership accuracy', 'mf_mia_accuracy_by_mask'),
    ('roc_auc', 'Missing-feature membership ROC-AUC', 'mf_mia_auc_by_mask'),
    ('tpr_at_1pct_fpr_roc', 'Missing-feature TPR at 1% FPR (ROC evaluation)', 'mf_mia_low_fpr_by_mask'),
]:
    plot_missing_feature_curves(metric, title, filename)

## Save final research results and interpretation limits
Save final CSV metrics, selected training histories, protocol, utility/complete-feature graphs and per-mask missing-feature graphs.
These research outputs include private-data evaluation statistics and are **not
DP-sanitized releases**. No private training rows, membership-labelled prediction
tables, fitted models, checkpoints or logs are written by this notebook.

The stated epsilon/delta bound concerns each individual target inference model
with its fixed public preprocessor. Releasing several models trained on the same
records requires privacy composition; the non-private control has no DP
guarantee. `secure_mode=False` is explicitly a research setting. Early stopping
uses public validation only. The recorded realised-stop accountant value is a
diagnostic; the fixed maximum-schedule bound is the claimed bound.

This study has one initialization seed per condition and independently sampled
DP noise. It uses transferred mixed tree/MLP-shadow attacks, native year-specific Census
codes, an auxiliary-derived vocabulary, record-level protection, and a fixed
research partition. Repeated runs and attacks matched to DP targets are follow-up
checks. Lower measured MIA does not by itself demonstrate stronger protection.

**Combined-study limitations.** A frozen non-private, cross-year shadow attack
can be weak against DP targets. Near-chance results do not prove that no stronger
attacker exists. Conditional completion is approximate, visible proxy fields
remain, and K=6 gives noisy std estimates. One seed and one independent DP noise
realization per budget provide exploratory evidence, not reliable repeatability
or optimal training settings. Uncertainty need not outperform mean-only, and
empirical curves need not be monotonic in epsilon.

Repeated queries and auxiliary-only completion/aggregation are post-processing
of each fixed DP inference model and do not themselves spend another training
privacy budget. This does not make the full sweep or its private-data research
metrics a DP release: per-model bounds, composition and the output limitations
above still apply. No attack query or target membership result influences training.

In [ ]:
def json_value(value):
    if isinstance(value, dict):
        return {str(key): json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_value(item) for item in value]
    if isinstance(value, np.generic):
        return json_value(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return str(value)
    return value

RUN_DIR.mkdir(parents=True, exist_ok=True)
study_results.to_csv(RUN_DIR / 'dp_target_and_mia_metrics.csv', index=False)
UNCERTAINTY_DIR.mkdir(parents=True, exist_ok=True)
missing_feature_results.to_csv(RUN_DIR / 'dp_missing_feature_metrics.csv', index=False)
uncertainty_changes.to_csv(RUN_DIR / 'uncertainty_vs_mean_only.csv', index=False)
uncertainty_auxiliary_results.to_csv(UNCERTAINTY_DIR / 'auxiliary_comparison.csv', index=False)
reconstruction_audit.to_csv(UNCERTAINTY_DIR / 'reconstruction_audit.csv', index=False)
public_roles = [dict(record_id=record_id, role=role)
    for role, ids in uncertainty_roles.items() for record_id in sorted(ids)]
pd.DataFrame(public_roles).to_csv(UNCERTAINTY_DIR / 'public_auxiliary_roles.csv', index=False)
auxiliary_attack_metrics.to_csv(RUN_DIR / 'auxiliary_attack_metrics.csv')
shadow_income_metrics.to_csv(RUN_DIR / 'shadow_income_metrics.csv')
for condition, history in training_histories.items():
    pd.DataFrame(history).to_csv(RUN_DIR / f'{condition}_public_validation_history.csv', index=False)
protocol = dict(study_variant='uncertainty-aware missing-feature transfer MIA against DP targets',
    accounting_calibration='Exact private-loader steps; conservative Opacus accountant rate',
    execution_status='complete', target_year=TARGET_YEAR, auxiliary_year=AUXILIARY_YEAR,
    epsilon_budgets=EPSILON_BUDGETS, delta=DELTA, train_size=TRAIN_SIZE,
    feature_count=len(EXPANDED_FEATURES), features=EXPANDED_FEATURES,
    label='nominal PINCP > 50000', split_config=SPLIT_CONFIG, target_partition_sizes=partition_counts,
    public_preprocessing_records=PUBLIC_PREPROCESSING_SIZE,
    preprocessing_scope='Fixed public 2014 subset, disjoint from shadow pool; shared by all targets',
    public_validation='Fixed 2023 validation split outside protected training records',
    training_config=TRAIN_CONFIG, target_results=target_metadata,
    privacy_scope='Individual target model; record-level DP conditional on fixed partition/public data',
    privacy_bound='Fixed maximum-schedule epsilon; epsilon at adaptive stop is diagnostic only',
    secure_mode=False, research_outputs_are_dp_releases=False,
    multiple_models='Per-model bounds; joint release requires composition; control is non-private',
    n_shadow_models=N_SHADOW_MODELS, shadow_algorithms=SHADOW_ALGORITHMS, shadow_seeds=shadow_seeds,
    shadow_models_per_algorithm=4, shadow_mlp_config=SHADOW_MLP_CONFIG,
    records_per_class_per_shadow=SHADOW_RECORDS_PER_CLASS,
    attack='Frozen complete-feature RF transfer attack; non-DP tree and generic MLP shadows',
    attack_features=list(attack_result['classifier'].feature_names_in_), include_blind_walk=True,
    walk_config=WALK_CONFIG, extra_numeric_protocol=EXPANDED_NUMERIC_WALK,
    walk_scope='Consistent extra numeric protocol for shadows and targets',
    shadow_query_audits=shadow_query_audits, auxiliary_attack_metrics=attack_result['metrics'],
    accuracy_threshold=attack_result['accuracy_threshold'], low_fpr_threshold=attack_result['low_fpr_threshold'],
    plots='Equally spaced categorical requested epsilon labels; non-private horizontal reference',
    versions=versions, checkpoints_enabled=False, logging_enabled=False,
    missing_feature_experiments=True, repeated_seed_confirmation=False,
    missing_feature_protocol=dict(k=UNCERTAINTY_K, masks=MISSING_MASKS,
        reconstruction_seed=RECONSTRUCTION_SEED, pfv_seed=SEED,
        reconstruction='2014-only conditional forest-leaf donors; no income or membership conditioning',
        reconstruction_chain_order=MASK_FIELDS, score_std_ddof=0,
        auxiliary_roles={name: len(ids) for name, ids in uncertainty_roles.items()},
        reconstruction_fit_records=len(recon_fit_ids), reconstruction_audit_records=len(recon_check_ids),
        public_preprocessing_records_excluded_from_donors=True,
        methods=['pfv', 'single', 'average', 'learned_mean', 'uncertainty'],
        conditional_thresholds=frozen_thresholds,
        calibration='auxiliary only; frozen across all target budgets',
        auxiliary_completion_query_audits=uncertainty_query_audits,
        target_membership_labels_used_for_fitting=False,
        target_mia_records='unchanged Exp 4 balanced member/nonmember partitions',
        primary_contrast='uncertainty minus learned_mean on identical conditional draws',
        source_exp3='exp_3_expanded_missing_feature_baseline_mlp.ipynb',
        source_exp4='exp_4_differential_privacy_accuracy.ipynb'),
    missing_feature_plots='Per-mask equally spaced requested epsilon categories plus non-private control')
_ = (RUN_DIR / 'protocol.json').write_text(json.dumps(json_value(protocol), indent=2, allow_nan=False), encoding='utf-8')
print('Final metrics:', RUN_DIR / 'dp_target_and_mia_metrics.csv')
print('Graphs: target_accuracy_vs_epsilon and mia_accuracy_vs_epsilon (PNG and SVG)')

print('Missing-feature metrics:', RUN_DIR / 'dp_missing_feature_metrics.csv')

Implementation references:
[Opacus PrivacyEngine](https://opacus.ai/api/privacy_engine.html),
[BatchMemoryManager](https://opacus.ai/api/batch_memory_manager.html),
[DPDataLoader](https://opacus.ai/api/data_loader.html),
[Opacus FAQ](https://opacus.ai/docs/faq).